# S10 - Actividad autónoma 4.1: series de tiempo e inferencia en streaming sobre el Proyecto Sello (equipo LLSW3)

**Proyecto Sello:** Sistema de monitoreo Big Data del tráfico de red del campus universitario (UPeU Juliaca).

## Fase 1 — Business Understanding

- **Qué se predice:** `bytesRecibidosSeg` del **segundo siguiente** (t+1) de una tarjeta de red del nodo de captura — la dimensión "volumen de tráfico" del Proyecto Sello.
- **Fuente:** el historial real del topic `red-telemetria` (lecturas del agente desde S7 a hoy, una por segundo y por tarjeta física).
- **Contra qué se compara:** el baseline de **persistencia** ("el próximo segundo será igual a este"). Un modelo solo se justifica si le gana con margen.
- **Inferencia en vivo:** sin estado — cada lectura que llega trae todo lo que el modelo necesita (sus propias tasas actuales).

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("s10-proyecto-red-series-tiempo")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4041")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("FATAL")
KAFKA, TOPIC = "kafka:9092", "red-telemetria"
spark.version

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8445092a-1674-462f-b1f9-808a7cf0abe0;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central


	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 292ms :: artifacts dl 11ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apa

26/10/17 20:01:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'4.2.0'

## Fase 2 — Data Understanding: historial real de `red-telemetria`

In [2]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_red = (
    StructType()
    .add("tipoEvento", StringType()).add("sensorId", StringType()).add("interfaz", StringType())
    .add("tipoInterfaz", StringType()).add("velocidadEnlaceMbps", LongType())
    .add("bytesRecibidosSeg", DoubleType()).add("bytesEnviadosSeg", DoubleType())
    .add("paquetesRecibidosSeg", DoubleType()).add("paquetesEnviadosSeg", DoubleType())
    .add("erroresEntrada", LongType()).add("erroresSalida", LongType())
    .add("descartesEntrada", LongType()).add("descartesSalida", LongType())
    .add("intervaloMs", LongType()).add("origen", StringType()).add("timestamp", LongType())
)
TARJETAS = ["llsw3-wi-fi-2", "llsw3-ethernet-2"]

historial = (
    spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("subscribe", TOPIC)
    .option("startingOffsets", "earliest").load()
    .select(F.from_json(F.col("value").cast("string"), esquema_red).alias("e")).select("e.*")
    .where(F.col("tipoEvento") == "red.lectura")
    .where(F.col("sensorId").isin(TARJETAS))
    .where((F.col("bytesRecibidosSeg") >= 0) & (F.col("bytesEnviadosSeg") >= 0))  # sin contadores reiniciados
    .withColumn("fechaHora", (F.col("timestamp") / 1000).cast("timestamp"))
)
print("lecturas reales de las tarjetas físicas:", historial.count())
historial.groupBy("sensorId").agg(
    F.count("*").alias("lecturas"), F.min("fechaHora").alias("desde"), F.max("fechaHora").alias("hasta"),
    F.round(F.avg("bytesRecibidosSeg")).alias("rxPromedio"), F.round(F.max("bytesRecibidosSeg")).alias("rxMax"),
).show(truncate=False)

lecturas reales de las tarjetas físicas: 6526


+----------------+--------+-----------------------+-----------------------+----------+---------+
|sensorId        |lecturas|desde                  |hasta                  |rxPromedio|rxMax    |
+----------------+--------+-----------------------+-----------------------+----------+---------+
|llsw3-ethernet-2|190     |2026-10-17 19:58:28.237|2026-10-17 20:01:46.889|1245714.0 |3325495.0|
|llsw3-wi-fi-2   |6338    |2026-10-13 14:51:33.829|2026-10-13 16:53:40.646|51736.0   |8715532.0|
+----------------+--------+-----------------------+-----------------------+----------+---------+



## Fase 3 — Data Preparation

### `label` con `lead()` sobre el orden cronológico

`label` = `bytesRecibidosSeg` de la **lectura siguiente de la misma tarjeta**. Como el historial tiene huecos entre
sesiones (días sin el agente corriendo), solo se conservan pares donde la siguiente lectura llegó **a lo sumo 3 s
después**: un `lead()` que salta de un día a otro no es "el segundo siguiente".

In [3]:
orden = Window.partitionBy("sensorId").orderBy("timestamp")
FEATURES = ["bytesRecibidosSeg", "bytesEnviadosSeg", "paquetesRecibidosSeg", "paquetesEnviadosSeg", "velocidadEnlaceMbps"]

dataset_ml = (
    historial
    .withColumn("label", F.lead("bytesRecibidosSeg", 1).over(orden))
    .withColumn("tsSiguiente", F.lead("timestamp", 1).over(orden))
    .where(F.col("label").isNotNull() & ((F.col("tsSiguiente") - F.col("timestamp")) <= 3000))
    .select("sensorId", "fechaHora", "timestamp", *FEATURES, "label")
)
print("filas del dataset supervisado:", dataset_ml.count())
dataset_ml.orderBy("timestamp").select("sensorId", "fechaHora", "bytesRecibidosSeg", "label").show(6, truncate=False)

filas del dataset supervisado: 6525


+-------------+-----------------------+-----------------+--------+
|sensorId     |fechaHora              |bytesRecibidosSeg|label   |
+-------------+-----------------------+-----------------+--------+
|llsw3-wi-fi-2|2026-10-13 14:51:33.829|4465.6           |512440.3|
|llsw3-wi-fi-2|2026-10-13 14:51:34.913|512440.3         |377132.7|
|llsw3-wi-fi-2|2026-10-13 14:51:36.021|377132.7         |8474.1  |
|llsw3-wi-fi-2|2026-10-13 14:51:37.121|8474.1           |427.1   |
|llsw3-wi-fi-2|2026-10-13 14:51:38.157|427.1            |2116.6  |
|llsw3-wi-fi-2|2026-10-13 14:51:39.287|2116.6           |1890.1  |
+-------------+-----------------------+-----------------+--------+
only showing top 6 rows


### División cronológica 80/20 (nunca aleatoria)

In [4]:
corte_ts = dataset_ml.approxQuantile("timestamp", [0.8], 0.0001)[0]
train = dataset_ml.where(F.col("timestamp") <= corte_ts).cache()
test = dataset_ml.where(F.col("timestamp") > corte_ts).cache()
print(f"train: {train.count():,} filas  |  test: {test.count():,} filas")
print("última fecha de train :", train.agg(F.max("fechaHora")).collect()[0][0])
print("primera fecha de test :", test.agg(F.min("fechaHora")).collect()[0][0])
print("¿hay solapamiento?    :", train.agg(F.max("timestamp")).collect()[0][0] >= test.agg(F.min("timestamp")).collect()[0][0])

train: 5,221 filas  |  test: 1,308 filas
última fecha de train : 2026-10-13 16:33:27.107000
primera fecha de test : 2026-10-13 16:33:28.184000


¿hay solapamiento?    : False


## Fase 4 — Modeling

### Baseline de persistencia (antes de cualquier modelo)

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

def metricas(df, nombre):
    r = {"modelo": nombre}
    for m in ("rmse", "mae", "r2"):
        r[m.upper() if m != "r2" else "R2"] = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m).evaluate(df)
    return r

pred_persistencia = test.withColumn("prediction", F.col("bytesRecibidosSeg"))
resultado_persistencia = metricas(pred_persistencia, "Persistencia (baseline)")
resultado_persistencia

{'modelo': 'Persistencia (baseline)',
 'RMSE': 162246.45596391225,
 'MAE': 48290.59655963311,
 'R2': 0.9220751446892508}

### `LinearRegression`

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor

ensamblador = VectorAssembler(inputCols=FEATURES, outputCol="features")
pipe_lr = Pipeline(stages=[ensamblador, LinearRegression(featuresCol="features", labelCol="label")])
modelo_lr = pipe_lr.fit(train)
resultado_lr = metricas(modelo_lr.transform(test), "LinearRegression")
print(resultado_lr)
print("coeficientes:", dict(zip(FEATURES, [round(c, 4) for c in modelo_lr.stages[-1].coefficients])))

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


{'modelo': 'LinearRegression', 'RMSE': 270302.2324974474, 'MAE': 113201.68340226654, 'R2': 0.7837157814290097}
coeficientes: {'bytesRecibidosSeg': np.float64(0.483), 'bytesEnviadosSeg': np.float64(0.0246), 'paquetesRecibidosSeg': np.float64(-459.1357), 'paquetesEnviadosSeg': np.float64(1596.6148), 'velocidadEnlaceMbps': np.float64(546.4453)}


### `RandomForestRegressor` y `GBTRegressor`

In [7]:
pipe_rf = Pipeline(stages=[ensamblador, RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)])
modelo_rf = pipe_rf.fit(train)
resultado_rf = metricas(modelo_rf.transform(test), "RandomForestRegressor")

pipe_gbt = Pipeline(stages=[ensamblador, GBTRegressor(featuresCol="features", labelCol="label", maxIter=40, maxDepth=5, seed=42)])
modelo_gbt = pipe_gbt.fit(train)
resultado_gbt = metricas(modelo_gbt.transform(test), "GBTRegressor")
print(resultado_rf); print(resultado_gbt)
print("importancias RF:", dict(zip(FEATURES, [round(float(v), 3) for v in modelo_rf.stages[-1].featureImportances])))

{'modelo': 'RandomForestRegressor', 'RMSE': 433354.113762992, 'MAE': 146993.67163584544, 'R2': 0.4440811328345706}
{'modelo': 'GBTRegressor', 'RMSE': 436049.32619891176, 'MAE': 147366.55838177947, 'R2': 0.4371446403106255}
importancias RF: {'bytesRecibidosSeg': 0.291, 'bytesEnviadosSeg': 0.168, 'paquetesRecibidosSeg': 0.274, 'paquetesEnviadosSeg': 0.263, 'velocidadEnlaceMbps': 0.004}


## Fase 5 — Evaluation: todos contra el baseline

In [8]:
import pandas as pd
comparacion = pd.DataFrame([resultado_persistencia, resultado_lr, resultado_rf, resultado_gbt])
base = resultado_persistencia["RMSE"]
comparacion["MejoraRMSE_vs_persistencia_%"] = ((base - comparacion["RMSE"]) / base * 100).round(2)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
comparacion

,modelo,RMSE,MAE,R2,MejoraRMSE_vs_persistencia_%
0,Persistencia (baseline),"162,246.456","48,290.597",0.922,0.000
1,LinearRegression,"270,302.232","113,201.683",0.784,-66.600
2,RandomForestRegressor,"433,354.114","146,993.672",0.444,-167.100
3,GBTRegressor,"436,049.326","147,366.558",0.437,-168.760


In [9]:
candidatos = comparacion[comparacion["modelo"] != "Persistencia (baseline)"]
ganador = candidatos.sort_values("RMSE").iloc[0]
mejora = ganador["MejoraRMSE_vs_persistencia_%"]
print(f"Modelo con menor RMSE: {ganador['modelo']}  (mejora vs persistencia: {mejora:.2f} %)")
if mejora <= 0:
    print("Ningún modelo le gana a la persistencia: se mantiene la regla simple en producción.")
elif mejora < 5:
    print("Gana por menos de 5 %: el margen no justifica mantener un modelo entrenado; se documenta y se prefiere la persistencia.")
else:
    print("Gana con margen suficiente: se despliega el modelo.")
modelos = {"LinearRegression": modelo_lr, "RandomForestRegressor": modelo_rf, "GBTRegressor": modelo_gbt}
modelo_ganador = modelos[ganador["modelo"]]

Modelo con menor RMSE: LinearRegression  (mejora vs persistencia: -66.60 %)
Ningún modelo le gana a la persistencia: se mantiene la regla simple en producción.


## Fase 6 — Deployment: guardar y recargar el `PipelineModel` (con su `VectorAssembler`)

In [10]:
from pyspark.ml import PipelineModel
RUTA_MODELO = "./artifacts/models/modelo_red_rx_t1"
modelo_ganador.write().overwrite().save(RUTA_MODELO)
modelo_cargado = PipelineModel.load(RUTA_MODELO)
print("etapas del modelo recargado:", [type(s).__name__ for s in modelo_cargado.stages])

antes = modelo_ganador.transform(test).select("timestamp", F.col("prediction").alias("p1"))
despues = modelo_cargado.transform(test).select("timestamp", F.col("prediction").alias("p2"))
dif = antes.join(despues, "timestamp").agg(F.max(F.abs(F.col("p1") - F.col("p2")))).collect()[0][0]
print("máxima diferencia de predicción antes/después de recargar:", dif)

etapas del modelo recargado: ['VectorAssembler', 'LinearRegressionModel']


máxima diferencia de predicción antes/después de recargar: 0.0


### Hallazgo: guardar solo el modelo, sin el `VectorAssembler`

Si se guarda únicamente la última etapa (el regresor) y se le pasa una lectura cruda, falla: no existe la columna
`features`. Por eso se despliega el `PipelineModel` completo.

In [11]:
solo_regresor = modelo_ganador.stages[-1]
try:
    solo_regresor.transform(test.limit(1)).show()
except Exception as ex:
    print(type(ex).__name__ + ":", str(ex).splitlines()[0][:220])

IllegalArgumentException: [FIELD_NOT_FOUND] No such struct field `features` in `sensorId`, `fechaHora`, `timestamp`, `bytesRecibidosSeg`, `bytesEnviadosSeg`, `paquetesRecibidosSeg`, `paquetesEnviadosSeg`, `velocidadEnlaceMbps`, `label`. SQLSTATE:


## Inferencia en tiempo real sobre `red-telemetria` (sin estado, sin ventanas)

Cada lectura que llega del agente se transforma con el `PipelineModel` y se guarda con su predicción t+1. Durante la
consulta el nodo descarga un archivo grande por HTTP para que haya tráfico real que predecir.

In [12]:
import shutil, threading, time, urllib.request
RUTA_PRED = "./artifacts/predicciones_red_stream"
CHK = "./artifacts/chk_predicciones_red_stream"
shutil.rmtree(RUTA_PRED, ignore_errors=True); shutil.rmtree(CHK, ignore_errors=True)

eventos = (
    spark.readStream.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("subscribe", TOPIC)
    .option("startingOffsets", "latest").option("failOnDataLoss", "false").load()
    .select(F.from_json(F.col("value").cast("string"), esquema_red).alias("e")).select("e.*")
    .where(F.col("sensorId").isin(TARJETAS))
)

def guardar(df, id_lote):
    filas = df.count()
    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas"); return
    (df.select(F.lit(id_lote).alias("batchId"), "sensorId", (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora"),
               "bytesRecibidosSeg", F.round("prediction", 1).alias("prediccionRxSiguiente"))
       .write.mode("append").parquet(RUTA_PRED))
    print(f"Micro-lote {id_lote}: {filas} predicciones guardadas")

def descarga_real(segundos):
    fin = time.time() + segundos
    try:
        with urllib.request.urlopen("https://proof.ovh.net/files/1Gb.dat", timeout=10) as r:
            while time.time() < fin and r.read(1 << 16):
                pass
    except Exception as ex:
        print("descarga:", ex)

consulta = modelo_cargado.transform(eventos).writeStream.foreachBatch(guardar).option("checkpointLocation", CHK).start()
time.sleep(25)
threading.Thread(target=descarga_real, args=(20,), daemon=True).start()
time.sleep(55)
consulta.stop()

Micro-lote 0: sin filas


Micro-lote 1: 1 predicciones guardadas


Micro-lote 2: 2 predicciones guardadas


Micro-lote 3: 1 predicciones guardadas


Micro-lote 4: 1 predicciones guardadas


...

Micro-lote 48: 2 predicciones guardadas


Micro-lote 49: 1 predicciones guardadas


Micro-lote 50: 1 predicciones guardadas


Micro-lote 51: 2 predicciones guardadas


Exception in thread "stream execution thread for [id = 67c9c067-87c1-477a-b0d5-743b5ab57f5b, runId = 7fc3c5ee-4b7b-49c9-9fff-a2916fc51cfe]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4106)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern

In [13]:
pred = spark.read.parquet(RUTA_PRED).orderBy("fechaHora")
w = Window.partitionBy("sensorId").orderBy("fechaHora")
real_vs_pred = pred.withColumn("rxRealSiguiente", F.lead("bytesRecibidosSeg").over(w))
print("predicciones en vivo guardadas:", pred.count())
real_vs_pred.select("batchId", "sensorId", "fechaHora", "bytesRecibidosSeg", "prediccionRxSiguiente", "rxRealSiguiente").show(25, truncate=False)

predicciones en vivo guardadas: 74


+-------+----------------+-----------------------+-----------------+---------------------+---------------+
|batchId|sensorId        |fechaHora              |bytesRecibidosSeg|prediccionRxSiguiente|rxRealSiguiente|
+-------+----------------+-----------------------+-----------------+---------------------+---------------+
|1      |llsw3-ethernet-2|2026-10-17 20:02:09.265|1463.3           |513191.4             |1220.8         |
|2      |llsw3-ethernet-2|2026-10-17 20:02:10.365|1220.8           |511682.3             |3772.4         |
|2      |llsw3-ethernet-2|2026-10-17 20:02:11.393|3772.4           |532715.1             |11168.8        |
|3      |llsw3-ethernet-2|2026-10-17 20:02:12.46 |11168.8          |510603.4             |1448.9         |
|4      |llsw3-ethernet-2|2026-10-17 20:02:13.488|1448.9           |506512.0             |1023.9         |
|5      |llsw3-ethernet-2|2026-10-17 20:02:14.54 |1023.9           |518334.2             |3218.7         |
|5      |llsw3-ethernet-2|2026-10-17 